In [2]:
import re
import pandas as pd
import numpy as np

file_path = "DataCo_Supply_Chain_Dataset.csv"

df = pd.read_csv(file_path, encoding="utf-8-sig")
df.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [4]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.info()

Rows: 180519
Columns: 53
<class 'pandas.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  str    
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  str    
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  str    
 9   Customer City                  180519 non-null  str    
 10  Customer Country               180519 non-null  str    
 11  Customer Email                 180519 non-null  str    
 12  Customer Fname  

In [6]:
# Clean column names for SQL
df.columns = [
    re.sub(r"[^\w]+", "_", c.strip().lower()).strip("_")
    for c in df.columns
]
print("\nColumns:", df.columns.tolist())


Columns: ['type', 'days_for_shipping_real', 'days_for_shipment_scheduled', 'benefit_per_order', 'sales_per_customer', 'delivery_status', 'late_delivery_risk', 'category_id', 'category_name', 'customer_city', 'customer_country', 'customer_email', 'customer_fname', 'customer_id', 'customer_lname', 'customer_password', 'customer_segment', 'customer_state', 'customer_street', 'customer_zipcode', 'department_id', 'department_name', 'latitude', 'longitude', 'market', 'order_city', 'order_country', 'order_customer_id', 'order_date_dateorders', 'order_id', 'order_item_cardprod_id', 'order_item_discount', 'order_item_discount_rate', 'order_item_id', 'order_item_product_price', 'order_item_profit_ratio', 'order_item_quantity', 'sales', 'order_item_total', 'order_profit_per_order', 'order_region', 'order_state', 'order_status', 'order_zipcode', 'product_card_id', 'product_category_id', 'product_description', 'product_image', 'product_name', 'product_price', 'product_status', 'shipping_date_dateo

In [8]:
# Check missing values
missing = df.isnull().sum()

print("Missing Values:")
print(missing[missing > 0])

Missing Values:
customer_lname              8
customer_zipcode            3
order_zipcode          155679
product_description    180519
dtype: int64


In [10]:
# Remove Duplicates
print("Duplicate Rows:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

print("Rows After Removing Duplicates:", len(df))
print("Duplicate order_item_id:", df["order_item_id"].duplicated().sum())

Duplicate Rows: 0
Rows After Removing Duplicates: 180519
Duplicate order_item_id: 0


In [12]:
# Drop unusable columns: masked (XXXX), URLs, and fully empty columns
def is_masked(s):
    """True if every non-null value is made only of X or * characters."""
    v = s.dropna().astype(str).str.strip()
    return len(v) > 0 and v.str.fullmatch(r"[Xx*]+").all()

def is_url(s):
    """True if almost all non-null values look like web links."""
    v = s.dropna().astype(str).str.strip()
    return len(v) > 0 and v.str.match(r"https?://", case=False).mean() > 0.9

text_cols   = df.select_dtypes(include=["object", "string"]).columns
masked_cols = [c for c in text_cols if is_masked(df[c])]
url_cols    = [c for c in text_cols if is_url(df[c])]
empty_cols  = df.columns[df.isnull().all()].tolist()

print("Masked:", masked_cols)
print("URL:", url_cols)
print("Empty:", empty_cols)

df = df.drop(columns=set(masked_cols + url_cols + empty_cols))
df.shape

Masked: ['customer_email', 'customer_password']
URL: ['product_image']
Empty: ['product_description']


(180519, 49)

In [14]:
# Handle missing values
# Zipcodes are identifiers, not numbers: convert to text, fill gaps with "Unknown"
for col in ["customer_zipcode", "order_zipcode"]:
    if col in df.columns:
        df[col] = (pd.to_numeric(df[col], errors="coerce")
                     .astype("Int64").astype("string").fillna("Unknown"))

# Text columns with missing values (only if the column still exists)
for col in ["customer_lname"]:
    if col in df.columns:
        df[col] = df[col].astype("string").fillna("Unknown")

print("Remaining Missing Values:", df.isnull().sum().sum())

Remaining Missing Values: 0


In [16]:
# Fix Data Types 
date_columns = [
    "order_date_dateorders",
    "shipping_date_dateorders"
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")

print(df[date_columns].dtypes)


order_date_dateorders       datetime64[us]
shipping_date_dateorders    datetime64[us]
dtype: object


In [18]:
# Clean Text Data 
text_columns = df.select_dtypes(include=["object", "string"]).columns

for col in text_columns:
    df[col] = df[col].astype("string").str.strip()

print("Text data cleaned successfully.")


Text data cleaned successfully.


In [20]:
# Quick Validation 
print("\nInvalid Dates:", df[date_columns].isnull().sum().to_dict())

print("Shipping before Order:",
      (df["shipping_date_dateorders"] < df["order_date_dateorders"]).sum())

for col in ["order_item_quantity", "order_item_product_price", "sales"]:
    print(f"Negative {col}:", (df[col] < 0).sum())


Invalid Dates: {'order_date_dateorders': 0, 'shipping_date_dateorders': 0}
Shipping before Order: 0
Negative order_item_quantity: 0
Negative order_item_product_price: 0
Negative sales: 0


In [22]:
print("Final Rows:", df.shape[0])
print("Final Columns:", df.shape[1])
print("Missing Values:", df.isnull().sum().sum())
print("Duplicate Rows:", df.duplicated().sum())

Final Rows: 180519
Final Columns: 49
Missing Values: 0
Duplicate Rows: 0


In [24]:
# Cleaned Dataset 
output_file = "DataCo_Supply_Chain_Cleaned.csv"

df.to_csv(output_file, index=False, encoding="utf-8-sig",date_format="%Y-%m-%d %H:%M:%S")

print("Cleaned dataset saved successfully:", output_file)

Cleaned dataset saved successfully: DataCo_Supply_Chain_Cleaned.csv


In [26]:
cleaned_df = pd.read_csv(output_file, parse_dates=date_columns, low_memory=False)

print("Reloaded Shape:", cleaned_df.shape)
print("Duplicates:", cleaned_df.duplicated().sum())
print("Missing Values:", cleaned_df.isnull().sum().sum())
cleaned_df.head()

Reloaded Shape: (180519, 49)
Duplicates: 0
Missing Values: 0


,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,...,order_state,order_status,order_zipcode,product_card_id,product_category_id,product_name,product_price,product_status,shipping_date_dateorders,shipping_mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,Java Occidental,COMPLETE,Unknown,1360,73,Smart watch,327.75,0,2018-02-03 22:56:00,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,Rajastαn,PENDING,Unknown,1360,73,Smart watch,327.75,0,2018-01-18 12:27:00,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,Rajastαn,CLOSED,Unknown,1360,73,Smart watch,327.75,0,2018-01-17 12:06:00,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,Queensland,COMPLETE,Unknown,1360,73,Smart watch,327.75,0,2018-01-16 11:45:00,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,Queensland,PENDING_PAYMENT,Unknown,1360,73,Smart watch,327.75,0,2018-01-15 11:24:00,Standard Class
